## Week 11: DBSCAN and HAC Clustering

This week I am going to be applying DBSCAN and HAC Clustering to my datasets to see how the data clusters together

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics import roc_curve, auc, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

### Dataset 1: Instacart Dataset

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [3]:
X = data1_transformed.drop(columns = ['ordered', 'user_id', 'product_id'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [ ]:
dbscan_results = {}

for eps in [0.3, 0.5, 0.7, 1.0]: 

    for min_samples in [5, 10, 20]:

        model = DBSCAN(eps = eps, min_samples = min_samples)

        train_clusters = model.fit_predict(X_train_scaled)

        n_clusters = len(set(train_clusters)) - (1 if -1 in train_clusters else 0)

        noise_pct = np.mean(train_clusters == -1)

        if n_clusters >=2:

            mask = train_clusters != -1

            silhouette = silhouette_score(X_train_scaled[mask], train_clusters[mask], sample_size = min(250, len(X_train_scaled)))

        else:
            silhouette = np.nan

        dbscan_results[(eps, min_samples)] = {'clusters' : n_clusters, 'noise_pct' : noise_pct, 
        'silhouette' : silhouette}

dbscan_results = pd.DataFrame(dbscan_results).T

dbscan_results

: 

In [ ]:
dbscan = DBSCAN(eps = 0.5, min_samples = 5)

train_clusters = dbscan.fit_predict(X_train_scaled)

core_points = dbscan.components_

core_labels = train_clusters[dbscan.core_sample_indices_]

nn = NearestNeighbors(n_neighbors = 1)

nn.fit(core_points)

distances, indices = nn.kneighbors(X_test_scaled)

test_clusters = core_labels[indices.ravel()].copy()

test_clusters[distances.ravel()[:,] > dbscan.eps] = -1

dbscan_profiles = X_test.copy()

dbscan_profiles['cluster'] = test_clusters

dbscan_profiles.groupby('cluster').mean()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
cluster,,,,,,,,,,,,,,,,,,,,,
-1,0.817746,0.342926,0.251799,0.235012,0.153477,0.139089,0.095923,0.105516,0.098321,0.083933,...,0.026379,0.021583,0.021583,0.021583,0.021583,0.293687,1.693017,2.833714,0.875154,1.760815
0,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.740466,2.955472,0.693147,2.124248
1,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.731616,2.879190,0.693147,1.031035
2,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.715928,2.991295,0.693147,0.881157


In [ ]:
dbscan_profiles['target'] = y_test.to_numpy()

dbscan_profiles.groupby('cluster')['target'].mean()

NameError: name 'y_train' is not defined

In [4]:
hac_scores = {}

for k in [2, 3, 4, 5, 6, 8, 10]:

    model = AgglomerativeClustering(n_clusters = k, linkage = 'ward')

    train_clusters = model.fit_predict(X_train_scaled)

    silhouette = silhouette_score(X_train_scaled, train_clusters, sample_size = min(2000, len(X_train_scaled)))

    hac_scores[k] = silhouette

pd.Series(hac_scores).sort_values(ascending = False)

plt.plot(list(hac_scores.keys()), list(hac_scores.values()), marker = 'o', color = 'violet')

plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('HAC Silhouette Scores')
plt.show()

MemoryError: Unable to allocate 218. GiB for an array with shape (29240511706,) and data type float64

### Dataset 2: Amazon Reviews Dataset

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked', 'rating'])
y = data2_transformed['liked']

X.head()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,False,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,True,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,True,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,True,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,True,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)